# Supervised vs unsupervised autoencoders on CICIDS2018: several training seeds in Google Colab

Repeats Phases 3, 4 and 6 with different random seeds (model start values, batch order and the 1,000,000 training rows change; the data split and test set stay the same) and then computes means, standard deviations and 95% intervals for Table 1 and Table 5.

**Before you start** (same Drive folder as before, `My Drive/cicids2018-ae-share`)
1. Upload the **new** `sup_vs_unsup_code_package.zip` from the project folder, replacing the old one (it now contains `run_seeds.py` and `src/aggregate_seeds.py`).
2. Keep the `processed` sub-folder with the 8 data files as it is.
3. Use a **CPU** runtime. Keep this tab open while a seed runs.

**Time:** about 65 minutes per seed (Colab CPU). Each seed has its own cell below and saves its results to Drive when it finishes, so a disconnect only loses the seed that was running. Start with seeds 1 and 2; run more later (finished seeds are never re-run). More seeds give narrower intervals: with 3 or 4 runs the 95% interval of the mean is still wide.

The earlier Colab run (seed 42) in your Drive folder `rerun_...` is added automatically as one more observation.

## 1. Connect Google Drive and check the files

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
SHARE = "/content/drive/MyDrive/cicids2018-ae-share"

EXPECTED = {
    "X_train.npy": 2142261632, "X_val.npy": 337985120, "X_test.npy": 623189504,
    "meta_train.pkl": 642637073, "meta_val.pkl": 101269873, "meta_test.pkl": 186810698,
    "scaler.json": 6134, "phase2_summary.json": 8863,
}
ok = os.path.exists(f"{SHARE}/sup_vs_unsup_code_package.zip")
print("code zip:", "found" if ok else "MISSING")
for name, size in EXPECTED.items():
    path = next((p for p in (f"{SHARE}/processed/{name}", f"{SHARE}/processed/Copy of {name}") if os.path.exists(p)), f"{SHARE}/processed/{name}")
    have = os.path.getsize(path) if os.path.exists(path) else None
    status = "ok" if have == size else ("MISSING" if have is None else f"WRONG SIZE ({have} bytes, expected {size})")
    ok &= have == size
    print(f"{name:22s} {status}")
assert ok, "Fix the files in Drive (see the list above) and run this cell again."

## 2. Unpack the code, copy the data, and restore seeds finished earlier (a few minutes)

In [ ]:
%cd /content
!rm -rf /content/cicids2018-ae
!unzip -q -o "{SHARE}/sup_vs_unsup_code_package.zip" -d /content
!mkdir -p /content/cicids2018-ae/data/processed
!cp -r "{SHARE}/processed/." /content/cicids2018-ae/data/processed/
%cd /content/cicids2018-ae

import os, shutil
d = "/content/cicids2018-ae/data/processed"
for f in os.listdir(d):
    if f.startswith("Copy of "):
        os.rename(f"{d}/{f}", f"{d}/{f[len('Copy of '):]}")
print(sorted(os.listdir(d)))
assert os.path.exists("run_seeds.py") and os.path.exists("src/aggregate_seeds.py"), "Upload the NEW sup_vs_unsup_code_package.zip to Drive (it must contain run_seeds.py)."

# bring back the seeds that finished in earlier sessions
if os.path.isdir(f"{SHARE}/seeds_results"):
    shutil.copytree(f"{SHARE}/seeds_results", "seeds", dirs_exist_ok=True)
    print("restored:", sorted(os.listdir("seeds")))

## 3. Install the exact package versions (a few minutes)

`virtualenv` is used because Python's own `venv` fails on Colab.

In [ ]:
!pip install -q virtualenv
!python -m virtualenv /content/venv
!/content/venv/bin/pip install -q numpy==2.5.2 pandas==3.0.5 scikit-learn==1.9.0 matplotlib==3.11.1 psutil==7.2.2 river==0.26.1
!/content/venv/bin/pip install -q torch --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple

## 4. Check that everything loads (expected: 2,163,852 test rows)

In [ ]:
%env MPLBACKEND=Agg
!/content/venv/bin/python -c "import torch, pandas, numpy, sklearn, scipy; print('torch', torch.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__, '| sklearn', sklearn.__version__); m = pandas.read_pickle('data/processed/meta_test.pkl'); print('meta_test rows:', len(m)); assert len(m) == 2163852, 'unexpected number of test rows'; print('OK')"

## 5. Show the commands for seeds 1 to 4 (nothing is trained)

In [ ]:
!/content/venv/bin/python run_seeds.py --seeds 1 2 3 4 --dry-run

## 6. Run one seed per cell (about 65 minutes each; keep this tab open)

Each cell trains the unsupervised, one-class and supervised models for one seed, then copies the results to Google Drive (`seeds_results`). If the session disconnects, reconnect, run sections 1 to 4 again and run the seed cell again; finished seeds are skipped.

In [ ]:
SEED = 1
%env MPLBACKEND=Agg
!/content/venv/bin/python run_seeds.py --seeds {SEED}
import shutil
shutil.copytree("seeds", f"{SHARE}/seeds_results", dirs_exist_ok=True, ignore=shutil.ignore_patterns("models"))
print("seed", SEED, "saved to Drive")

In [ ]:
SEED = 2
%env MPLBACKEND=Agg
!/content/venv/bin/python run_seeds.py --seeds {SEED}
import shutil
shutil.copytree("seeds", f"{SHARE}/seeds_results", dirs_exist_ok=True, ignore=shutil.ignore_patterns("models"))
print("seed", SEED, "saved to Drive")

In [ ]:
SEED = 3
%env MPLBACKEND=Agg
!/content/venv/bin/python run_seeds.py --seeds {SEED}
import shutil
shutil.copytree("seeds", f"{SHARE}/seeds_results", dirs_exist_ok=True, ignore=shutil.ignore_patterns("models"))
print("seed", SEED, "saved to Drive")

In [ ]:
SEED = 4
%env MPLBACKEND=Agg
!/content/venv/bin/python run_seeds.py --seeds {SEED}
import shutil
shutil.copytree("seeds", f"{SHARE}/seeds_results", dirs_exist_ok=True, ignore=shutil.ignore_patterns("models"))
print("seed", SEED, "saved to Drive")

## 7. Intervals over the seeds

Run this after at least two seeds have finished (more seeds give a more reliable interval). It adds the earlier Colab run (seed 42) from your Drive as one more observation.

In [ ]:
import glob
extra = sorted(glob.glob(f"{SHARE}/rerun_*/results"))
extra_arg = f"--extra 42={extra[-1]}" if extra else ""
print("extra run:", extra[-1] if extra else "none found")
!/content/venv/bin/python src/aggregate_seeds.py --seeds-dir seeds {extra_arg}

## 8. Show the summary table and save it to Drive

In [ ]:
from IPython.display import Markdown, display
import shutil

display(Markdown(open("seeds/seeds_summary.md", encoding="utf-8").read()))
for f in ("seeds_summary.md", "seeds_summary.json"):
    shutil.copy(f"seeds/{f}", f"{SHARE}/{f}")
print("saved seeds_summary.md and seeds_summary.json to", SHARE)